# SahelDust v2: Enriched Features with AOD Threshold 0.7

This notebook uses a higher AOD labelling threshold (0.7 instead of 0.5) to focus on significant dust events with clear atmospheric signatures. This removes borderline cases that no model can distinguish from non-events.

Features: 7 atmospheric variables over 72-hour sequences, 7 surface features including vegetation water content and previous-day AOD.

Hyperparameters are informed by Optuna optimization on the v1 dataset which found GRU encoder, BCE loss, embed_dim=256, num_heads=8, high dropout, and lower learning rate as the best configuration.

# NOTE
This notebook is designed to originally run on Kaggle. Kindly contact the author for access to the dataset.

## 1. Setup

In [ ]:
import numpy as np
import os
import time
import gc
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score,
    recall_score, average_precision_score, accuracy_score,
    roc_curve, precision_recall_curve, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score, log_loss
)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

N_ATM_VARS = 7
N_SURF_VARS = 7


def full_metrics(y_true, y_pred, y_probs, name):
    print(f"{name}:")
    print(f"precision={precision_score(y_true, y_pred):.4f}")
    print(f"recall={recall_score(y_true, y_pred):.4f}")
    print(f"f1={f1_score(y_true, y_pred):.4f}")
    print(f"acc={accuracy_score(y_true, y_pred):.4f}")
    print(f"bal_acc={balanced_accuracy_score(y_true, y_pred):.4f}")
    print(f"roc_auc={roc_auc_score(y_true, y_probs):.4f}")
    print(f"pr_auc={average_precision_score(y_true, y_probs):.4f}")
    print(f"mcc={matthews_corrcoef(y_true, y_pred):.4f}")
    print(f"kappa={cohen_kappa_score(y_true, y_pred):.4f}")
    print(f"log_loss={log_loss(y_true, y_probs):.4f}")


print("Setup completed")

## 1.1 Function to plot figures for the models

In [ ]:
def plot_model_performance(history, y_true, y_probs, y_preds, model_name, save_prefix=None):
    """
    Generates all evaluation figures for a trained model.
    Call this after each model finishes training.
    """
    fig = plt.figure(figsize=(24, 20))
    fig.suptitle(f'{model_name} Performance', fontsize=16, fontweight='bold', y=0.98)

    # 1. Training and Validation Loss
    ax1 = fig.add_subplot(3, 3, 1)
    ax1.plot(history['train_loss'], label='Train Loss', color='#1A5FA0', lw=2)
    ax1.plot(history['val_loss'], label='Val Loss', color='#B84020', lw=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training vs Validation Loss')
    ax1.legend()
    ax1.grid(alpha=0.3)

    # 2. Validation Precision, Recall, F1 over epochs
    ax2 = fig.add_subplot(3, 3, 2)
    ax2.plot(history['val_precision'], label='Precision', color='#0C6E52', lw=2)
    ax2.plot(history['val_recall'], label='Recall', color='#A06C10', lw=2)
    ax2.plot(history['val_f1'], label='F1', color='#4A42B8', lw=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Score')
    ax2.set_title('Validation Metrics Over Training')
    ax2.legend()
    ax2.grid(alpha=0.3)
    ax2.set_ylim(0, 1)

    # 3. ROC Curve
    ax3 = fig.add_subplot(3, 3, 3)
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    auc_val = roc_auc_score(y_true, y_probs)
    ax3.plot(fpr, tpr, color='#1A5FA0', lw=2, label=f'ROC-AUC = {auc_val:.4f}')
    ax3.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    ax3.set_xlabel('False Positive Rate')
    ax3.set_ylabel('True Positive Rate')
    ax3.set_title('ROC Curve')
    ax3.legend(loc='lower right')
    ax3.grid(alpha=0.3)

    # 4. Precision-Recall Curve
    ax4 = fig.add_subplot(3, 3, 4)
    prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = average_precision_score(y_true, y_probs)
    ax4.plot(rec_curve, prec_curve, color='#B84020', lw=2, label=f'PR-AUC = {pr_auc:.4f}')
    baseline = y_true.mean()
    ax4.axhline(y=baseline, color='k', linestyle='--', lw=1, alpha=0.5, label=f'Baseline = {baseline:.3f}')
    ax4.set_xlabel('Recall')
    ax4.set_ylabel('Precision')
    ax4.set_title('Precision-Recall Curve')
    ax4.legend(loc='upper right')
    ax4.grid(alpha=0.3)
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)

    # 5. Confusion Matrix
    ax5 = fig.add_subplot(3, 3, 5)
    cm = confusion_matrix(y_true, y_preds)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax5,
                xticklabels=['No Dust', 'Dust'], yticklabels=['No Dust', 'Dust'])
    ax5.set_xlabel('Predicted')
    ax5.set_ylabel('Actual')
    ax5.set_title('Confusion Matrix')

    # 6. Threshold Sensitivity
    ax6 = fig.add_subplot(3, 3, 6)
    thresholds = np.arange(0.1, 0.9, 0.02)
    t_prec, t_rec, t_f1 = [], [], []
    for t in thresholds:
        t_preds = (y_probs > t).astype(int)
        if t_preds.sum() == 0:
            t_prec.append(0)
            t_rec.append(0)
            t_f1.append(0)
            continue
        t_prec.append(precision_score(y_true, t_preds, zero_division=0))
        t_rec.append(recall_score(y_true, t_preds, zero_division=0))
        t_f1.append(f1_score(y_true, t_preds, zero_division=0))
    ax6.plot(thresholds, t_prec, label='Precision', color='#0C6E52', lw=2)
    ax6.plot(thresholds, t_rec, label='Recall', color='#A06C10', lw=2)
    ax6.plot(thresholds, t_f1, label='F1', color='#4A42B8', lw=2)
    ax6.axhline(y=0.6, color='red', linestyle='--', lw=1, alpha=0.7, label='Target P=0.6')
    ax6.set_xlabel('Classification Threshold')
    ax6.set_ylabel('Score')
    ax6.set_title('Threshold Sensitivity Analysis')
    ax6.legend(fontsize=8)
    ax6.grid(alpha=0.3)
    ax6.set_ylim(0, 1)

    # 7. Prediction Probability Distribution
    ax7 = fig.add_subplot(3, 3, 7)
    ax7.hist(y_probs[y_true == 0], bins=50, alpha=0.6, color='#1A5FA0',
             label='Non-dust', density=True)
    ax7.hist(y_probs[y_true == 1], bins=50, alpha=0.6, color='#B84020',
             label='Dust', density=True)
    ax7.set_xlabel('Predicted Probability')
    ax7.set_ylabel('Density')
    ax7.set_title('Prediction Distribution by Class')
    ax7.legend()
    ax7.grid(alpha=0.3)

    # 8. Calibration Plot
    ax8 = fig.add_subplot(3, 3, 8)
    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_means = []
    bin_true_rates = []
    bin_counts = []
    for i in range(n_bins):
        mask = (y_probs >= bin_edges[i]) & (y_probs < bin_edges[i + 1])
        if mask.sum() > 0:
            bin_means.append(y_probs[mask].mean())
            bin_true_rates.append(y_true[mask].mean())
            bin_counts.append(mask.sum())
        else:
            bin_means.append((bin_edges[i] + bin_edges[i + 1]) / 2)
            bin_true_rates.append(0)
            bin_counts.append(0)
    ax8.plot(bin_means, bin_true_rates, 'o-', color='#4A42B8', lw=2, label='Model')
    ax8.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Perfect calibration')
    ax8.set_xlabel('Mean Predicted Probability')
    ax8.set_ylabel('True Positive Rate')
    ax8.set_title('Calibration Plot')
    ax8.legend()
    ax8.grid(alpha=0.3)
    ax8.set_xlim(0, 1)
    ax8.set_ylim(0, 1)

    # 9. Summary Metrics Text Box
    ax9 = fig.add_subplot(3, 3, 9)
    ax9.axis('off')
    p = precision_score(y_true, y_preds, zero_division=0)
    r = recall_score(y_true, y_preds, zero_division=0)
    f = f1_score(y_true, y_preds, zero_division=0)
    acc = accuracy_score(y_true, y_preds)
    bacc = balanced_accuracy_score(y_true, y_preds)
    mcc = matthews_corrcoef(y_true, y_preds)
    kappa = cohen_kappa_score(y_true, y_preds)
    ll = log_loss(y_true, y_probs)

    metrics_text = (
        f"Test Set Metrics\n"
        f"{'='*30}\n"
        f"Precision:      {p:.4f}\n"
        f"Recall:         {r:.4f}\n"
        f"F1 Score:       {f:.4f}\n"
        f"Accuracy:       {acc:.4f}\n"
        f"Balanced Acc:   {bacc:.4f}\n"
        f"ROC-AUC:        {auc_val:.4f}\n"
        f"PR-AUC:         {pr_auc:.4f}\n"
        f"MCC:            {mcc:.4f}\n"
        f"Cohen Kappa:    {kappa:.4f}\n"
        f"Log Loss:       {ll:.4f}\n"
        f"{'='*30}\n"
        f"Test samples:   {len(y_true):,}\n"
        f"Dust rate:      {y_true.mean()*100:.1f}%"
    )
    ax9.text(0.1, 0.95, metrics_text, transform=ax9.transAxes,
             fontsize=11, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    if save_prefix:
        plt.savefig(f'{save_prefix}_performance.png', dpi=150, bbox_inches='tight')
        print(f"  Saved: {save_prefix}_performance.png")

    plt.show()
    plt.close()


print("plot_model_performance function defined")

## 2. Load Data

In [ ]:
for root, dirs, files in os.walk('/kaggle/input'):
    npy_files = [f for f in files if f.endswith('.npy')]
    if len(npy_files) >= 9:
        DATA_DIR = root
        print(f"Found data at: {DATA_DIR}")
        break

X_atm_train = np.load(f'{DATA_DIR}/X_atm_train.npy')
X_surf_train = np.load(f'{DATA_DIR}/X_surf_train.npy')
y_train = np.load(f'{DATA_DIR}/y_train.npy')

X_atm_val = np.load(f'{DATA_DIR}/X_atm_val.npy')
X_surf_val = np.load(f'{DATA_DIR}/X_surf_val.npy')
y_val = np.load(f'{DATA_DIR}/y_val.npy')

X_atm_test = np.load(f'{DATA_DIR}/X_atm_test.npy')
X_surf_test = np.load(f'{DATA_DIR}/X_surf_test.npy')
y_test = np.load(f'{DATA_DIR}/y_test.npy')

print(f"Train: {len(y_train):,} | dust rate: {y_train.mean()*100:.1f}%")
print(f"Val: {len(y_val):,} | dust rate: {y_val.mean()*100:.1f}%")
print(f"Test: {len(y_test):,} | dust rate: {y_test.mean()*100:.1f}%")
print(f"Atm: {X_atm_train.shape} | Surf: {X_surf_train.shape}")

## 3. Flat Features for Baselines

In [ ]:
def extract_flat_features(X_atm, X_surf):
    feat_mean = X_atm.mean(axis=1)
    feat_max = X_atm.max(axis=1)
    feat_std = X_atm.std(axis=1)
    wind_speed = np.sqrt(X_atm[:,:,0]**2 + X_atm[:,:,1]**2)
    ws_mean = wind_speed.mean(axis=1, keepdims=True)
    ws_max = wind_speed.max(axis=1, keepdims=True)
    ws_std = wind_speed.std(axis=1, keepdims=True)
    recent_mean = X_atm[:, -12:, :].mean(axis=1)
    flat_atm = np.concatenate(
        [feat_mean, feat_max, feat_std, ws_mean, ws_max, ws_std, recent_mean],
        axis=1)
    return np.concatenate([flat_atm, X_surf], axis=1).astype(np.float32)

X_flat_train = extract_flat_features(X_atm_train, X_surf_train)
X_flat_val = extract_flat_features(X_atm_val, X_surf_val)
X_flat_test = extract_flat_features(X_atm_test, X_surf_test)
print(f"Flat features: {X_flat_train.shape}")

## 4. Baseline Models

Trained on a 400K subsample for LR and RF (CPU models). XGBoost uses the full dataset on GPU.

In [ ]:
print("BASELINE MODELS")


os.makedirs('/kaggle/working/models', exist_ok=True)

np.random.seed(42)
sub_idx = np.random.choice(len(y_train), size=min(400000, len(y_train)), replace=False)
X_flat_sub = X_flat_train[sub_idx]
y_sub = y_train[sub_idx]
print(f"LR/RF subsample: {len(y_sub):,} | dust rate: {y_sub.mean()*100:.1f}%")

import joblib

print("\nLogistic Regression")
t0 = time.time()
best_lr = LogisticRegression(C=1.0, penalty='l2', solver='lbfgs',
                              class_weight='balanced', max_iter=1000, random_state=42)
best_lr.fit(X_flat_sub, y_sub)
lr_probs = best_lr.predict_proba(X_flat_test)[:, 1]
lr_preds = best_lr.predict(X_flat_test)
print(f"  Time: {time.time()-t0:.0f}s")
full_metrics(y_test, lr_preds, lr_probs, "Logistic Regression")
joblib.dump(best_lr, '/kaggle/working/models/lr_v2.pkl')

print("\nRandom Forest")
t0 = time.time()
best_rf = RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_leaf=20,
                                  max_features='sqrt', class_weight='balanced',
                                  random_state=42, n_jobs=-1)
best_rf.fit(X_flat_sub, y_sub)
rf_probs = best_rf.predict_proba(X_flat_test)[:, 1]
rf_preds = best_rf.predict(X_flat_test)
print(f"  Time: {time.time()-t0:.0f}s")
full_metrics(y_test, rf_preds, rf_probs, "Random Forest")
joblib.dump(best_rf, '/kaggle/working/models/rf_v2.pkl')

print("\nXGBoost")
t0 = time.time()
best_xgb = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8, scale_pos_weight=1.0,
                           device='cuda' if torch.cuda.is_available() else 'cpu',
                           eval_metric='aucpr', early_stopping_rounds=15,
                           random_state=42, verbosity=0)
best_xgb.fit(X_flat_train, y_train, eval_set=[(X_flat_val, y_val)], verbose=False)
xgb_probs = best_xgb.predict_proba(X_flat_test)[:, 1]
xgb_preds = best_xgb.predict(X_flat_test)
print(f"  Time: {time.time()-t0:.0f}s")
full_metrics(y_test, xgb_preds, xgb_probs, "XGBoost")
joblib.dump(best_xgb, '/kaggle/working/models/xgb_v2.pkl')

feature_names = [
    'u10_mean','v10_mean','t2m_mean','sp_mean','blh_mean','tp_mean','d2m_mean',
    'u10_max','v10_max','t2m_max','sp_max','blh_max','tp_max','d2m_max',
    'u10_std','v10_std','t2m_std','sp_std','blh_std','tp_std','d2m_std',
    'ws_mean','ws_max','ws_std',
    'u10_last12','v10_last12','t2m_last12','sp_last12','blh_last12','tp_last12','d2m_last12',
    'soil_moisture','veg_water_content','prev_day_aod','lat','lon','month_sin','month_cos'
]
importances = best_rf.feature_importances_
idx = np.argsort(importances)[::-1]
print("\nRF Feature Importances (top 10):")
for i in idx[:10]:
    print(f"{feature_names[i]:20s}: {importances[i]:.4f}")

### Baseline Models Plots

In [ ]:
# Baseline model plots (no training history)
for name, probs, preds in [
    ('Logistic Regression', lr_probs, lr_preds),
    ('Random Forest', rf_probs, rf_preds),
    ('XGBoost', xgb_probs, xgb_preds),
]:
    fig = plt.figure(figsize=(18, 5))
    fig.suptitle(f'{name} Performance', fontsize=14, fontweight='bold')

    ax1 = fig.add_subplot(1, 3, 1)
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax1.plot(fpr, tpr, color='#1A5FA0', lw=2,
             label=f'AUC = {roc_auc_score(y_test, probs):.4f}')
    ax1.plot([0, 1], [0, 1], 'k--', lw=1)
    ax1.set_title('ROC Curve')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2 = fig.add_subplot(1, 3, 2)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax2,
                xticklabels=['No Dust', 'Dust'], yticklabels=['No Dust', 'Dust'])
    ax2.set_title('Confusion Matrix')

    ax3 = fig.add_subplot(1, 3, 3)
    prec_c, rec_c, _ = precision_recall_curve(y_test, probs)
    ax3.plot(rec_c, prec_c, color='#B84020', lw=2,
             label=f'PR-AUC = {average_precision_score(y_test, probs):.4f}')
    ax3.set_title('PR Curve')
    ax3.legend()
    ax3.grid(alpha=0.3)

    plt.tight_layout()
    safe = name.lower().replace(' ', '_')
    plt.savefig(f'/kaggle/working/{safe}_v2_performance.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

## 5. Neural Network Components

Model architecture uses findings from Optuna optimization on the v1 dataset. The DustModel factory builds any combination of encoder type (CNN, LSTM, GRU) and fusion method (concatenation, cross-modal attention). The default configuration uses embed_dim=256 and num_heads=8 based on Optuna results. Overfitting is controlled through dropout, weight decay, gradient clipping, and early stopping.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * bce).mean()

class CrossModalAttention(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn_s2a = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.attn_a2s = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_s = nn.LayerNorm(embed_dim)
        self.norm_a = nn.LayerNorm(embed_dim)
    def forward(self, surf_emb, atm_emb):
        s, a = surf_emb.unsqueeze(1), atm_emb.unsqueeze(1)
        s_out, _ = self.attn_s2a(s, a, a)
        a_out, _ = self.attn_a2s(a, s, s)
        return self.norm_s(s_out.squeeze(1) + surf_emb), self.norm_a(a_out.squeeze(1) + atm_emb)

class CNNEncoder(nn.Module):
    def __init__(self, input_channels=7, embed_dim=256, dropout=0.24):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=7, padding=3),
            nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(dropout),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(dropout),
            nn.Conv1d(128, embed_dim, kernel_size=3, padding=1),
            nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        return self.norm(self.conv(x.transpose(1, 2)).squeeze(-1))

class LSTMEncoder(nn.Module):
    def __init__(self, input_dim=7, embed_dim=256, hidden_dim=128, num_layers=2, dropout=0.24):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.projection = nn.Linear(hidden_dim * 2, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.norm(self.projection(self.dropout(torch.cat([h_n[-2], h_n[-1]], dim=-1))))

class GRUEncoder(nn.Module):
    def __init__(self, input_dim=7, embed_dim=256, hidden_dim=128, num_layers=2, dropout=0.24):
        super().__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.projection = nn.Linear(hidden_dim * 2, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        _, h_n = self.gru(x)
        return self.norm(self.projection(self.dropout(torch.cat([h_n[-2], h_n[-1]], dim=-1))))

class SurfaceEncoder(nn.Module):
    def __init__(self, input_dim=7, embed_dim=256, dropout=0.24):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, embed_dim), nn.LayerNorm(embed_dim))
    def forward(self, x):
        return self.net(x)

class PredictionHead(nn.Module):
    def __init__(self, input_dim=512, dropout=0.54):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

class DustModel(nn.Module):
    def __init__(self, encoder_type='gru', fusion='attention',
                 embed_dim=256, dropout=0.24, num_heads=8):
        super().__init__()
        if encoder_type == 'cnn':
            self.atm_encoder = CNNEncoder(input_channels=N_ATM_VARS, embed_dim=embed_dim, dropout=dropout)
        elif encoder_type == 'lstm':
            self.atm_encoder = LSTMEncoder(input_dim=N_ATM_VARS, embed_dim=embed_dim, dropout=dropout)
        elif encoder_type == 'gru':
            self.atm_encoder = GRUEncoder(input_dim=N_ATM_VARS, embed_dim=embed_dim, dropout=dropout)
        self.surf_encoder = SurfaceEncoder(input_dim=N_SURF_VARS, embed_dim=embed_dim, dropout=dropout)
        self.fusion = fusion
        if fusion == 'attention':
            self.cross_attn = CrossModalAttention(embed_dim=embed_dim, num_heads=num_heads)
        self.head = PredictionHead(input_dim=embed_dim * 2, dropout=0.54)

    def forward(self, atm, surf):
        a = self.atm_encoder(atm)
        s = self.surf_encoder(surf)
        if self.fusion == 'attention':
            s, a = self.cross_attn(s, a)
        return self.head(torch.cat([s, a], dim=-1))

class DustDataset(Dataset):
    def __init__(self, X_atm, X_surf, y):
        self.X_atm = torch.FloatTensor(X_atm)
        self.X_surf = torch.FloatTensor(X_surf)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X_atm[idx], self.X_surf[idx], self.y[idx]

train_ds = DustDataset(X_atm_train, X_surf_train, y_train)
val_ds = DustDataset(X_atm_val, X_surf_val, y_val)
test_ds = DustDataset(X_atm_test, X_surf_test, y_test)
train_dl = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)
test_dl = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training batches: {len(train_dl)}")
print("All components defined")

## 6. Training Function

Includes gradient clipping, cosine annealing, early stopping, and validation-calibrated threshold selection. The threshold that maximises F1 on the validation set is found after training and applied to the test set, instead of using an arbitrary 0.5 cutoff.

In [ ]:
def train_and_evaluate(model, train_dl, val_dl, test_dl, criterion,
                       lr=0.00032, weight_decay=0.003, epochs=20, model_name="Model"):
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)

    best_f1 = 0
    best_state = None
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [],
               'val_precision': [], 'val_recall': []}

    print(f"\n{'Ep':>4} | {'TrLoss':>8} | {'VaLoss':>8} | {'P':>6} | {'R':>6} | {'F1':>6} | {'Acc':>6} | {'AUC':>6}")
    print("-" * 68)

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0
        for atm, surf, y in train_dl:
            atm, surf, y = atm.to(device), surf.to(device), y.to(device)
            optimiser.zero_grad()
            loss = criterion(model(atm, surf), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()
            train_loss += loss.item() * len(y)
        train_loss /= len(train_dl.dataset)

        model.eval()
        val_loss = 0
        vp, vt = [], []
        with torch.no_grad():
            for atm, surf, y in val_dl:
                atm, surf, y_d = atm.to(device), surf.to(device), y.to(device)
                logits = model(atm, surf)
                val_loss += criterion(logits, y_d).item() * len(y)
                vp.extend(torch.sigmoid(logits).cpu().numpy())
                vt.extend(y.numpy())
        val_loss /= len(val_dl.dataset)

        vp_arr = np.array(vp)
        preds = (vp_arr > 0.5).astype(int)
        p = precision_score(vt, preds, zero_division=0)
        r = recall_score(vt, preds, zero_division=0)
        f = f1_score(vt, preds, zero_division=0)
        acc = accuracy_score(vt, preds)
        auc = roc_auc_score(vt, vp_arr)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(f)
        history['val_precision'].append(p)
        history['val_recall'].append(r)

        if f > best_f1:
            best_f1 = f
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()

        if epoch % 5 == 0 or epoch == 1:
            print(f"{epoch:>4} | {train_loss:>8.4f} | {val_loss:>8.4f} | "
                  f"{p:>6.4f} | {r:>6.4f} | {f:>6.4f} | {acc:>6.4f} | {auc:>6.4f}")

    model.load_state_dict(best_state)
    print(f"Best Val F1: {best_f1:.4f}")

    # Find optimal threshold on validation set
    model.eval()
    val_probs_cal = []
    val_true_cal = []
    with torch.no_grad():
        for atm, surf, y in val_dl:
            atm, surf = atm.to(device), surf.to(device)
            val_probs_cal.extend(torch.sigmoid(model(atm, surf)).cpu().numpy())
            val_true_cal.extend(y.numpy())
    val_probs_cal = np.array(val_probs_cal)
    val_true_cal = np.array(val_true_cal)

    best_val_f1 = 0
    best_threshold = 0.5
    for t in np.arange(0.3, 0.7, 0.01):
        vp_t = (val_probs_cal > t).astype(int)
        vf = f1_score(val_true_cal, vp_t, zero_division=0)
        if vf > best_val_f1:
            best_val_f1 = vf
            best_threshold = t
    print(f"Calibrated threshold: {best_threshold:.2f} (val F1: {best_val_f1:.4f})")

    # Test evaluation with calibrated threshold
    model.eval()
    tp, tt = [], []
    with torch.no_grad():
        for atm, surf, y in test_dl:
            atm, surf = atm.to(device), surf.to(device)
            tp.extend(torch.sigmoid(model(atm, surf)).cpu().numpy())
            tt.extend(y.numpy())
    tp = np.array(tp)
    tt = np.array(tt)

    # Report both default and calibrated threshold
    td_default = (tp > 0.5).astype(int)
    td_calibrated = (tp > best_threshold).astype(int)

    print(f"\n{model_name} Test (threshold=0.50):")
    print(f"  P={precision_score(tt, td_default):.4f} R={recall_score(tt, td_default):.4f} "
          f"F1={f1_score(tt, td_default):.4f} Acc={accuracy_score(tt, td_default):.4f}")

    full_metrics(tt, td_calibrated, tp, f"{model_name} Test (threshold={best_threshold:.2f})")

    return model, tp, td_calibrated, tt, history, best_threshold

## 7. Neural Network Experiments

Six encoder-fusion combinations. Based on Optuna findings, all models use embed_dim=256, num_heads=8, encoder dropout=0.24, head dropout=0.54, lr=0.00032, weight_decay=0.003. BCE loss is used as Optuna found it outperforms Focal Loss on balanced training data. Each model is saved immediately after training.

In [ ]:
print("NEURAL NETWORK EXPERIMENTS (BCE LOSS)")

experiments = [
    ('cnn', 'concat', 'CNN + Concat'),
    ('cnn', 'attention', 'CNN + CMA'),
    ('lstm', 'concat', 'LSTM + Concat'),
    ('lstm', 'attention', 'LSTM + CMA'),
    ('gru', 'concat', 'GRU + Concat'),
    ('gru', 'attention', 'GRU + CMA'),
]

nn_results = {}
all_histories = {}
all_thresholds = {}
criterion = nn.BCEWithLogitsLoss()

for enc, fus, name in experiments:
    print(f"\n{'=' * 60}")
    print(f"EXPERIMENT: {name}")
    print(f"{'=' * 60}")

    model = DustModel(encoder_type=enc, fusion=fus,
                      embed_dim=256, dropout=0.24, num_heads=8).to(device)
    params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {params:,}")

    model, probs, preds, true, history, threshold = train_and_evaluate(
        model, train_dl, val_dl, test_dl,
        criterion, lr=0.00032, weight_decay=0.003,
        epochs=20, model_name=name
    )

    nn_results[name] = {'probs': probs, 'preds': preds, 'model': model, 'params': params}
    all_histories[name] = history
    all_thresholds[name] = threshold

    safe = name.lower().replace(' ', '_').replace('+', '_')
    torch.save(model.state_dict(), f'/kaggle/working/models/{safe}_v2.pt')
    print(f"  Saved: {safe}_v2.pt")

    plot_model_performance(
        history, true, probs, preds, name,
        save_prefix=f'/kaggle/working/{safe}_v2'
    )

    torch.cuda.empty_cache()
    gc.collect()

## 8. Focal Loss Comparison

The Optuna search on v1 data found BCE optimal. Testing whether Focal Loss performs differently on the v2 (0.7 threshold) data which has a different class distribution.

In [ ]:
print("FOCAL LOSS VARIANTS (LSTM + CMA only)")

focal_variants = [
    (0.25, 2.0, 'Focal a=0.25 g=2.0'),
    (0.40, 2.0, 'Focal a=0.40 g=2.0'),
    (0.25, 3.0, 'Focal a=0.25 g=3.0'),
]

for alpha, gamma, label in focal_variants:
    name = f'LSTM + CMA + {label}'
    print(f"\n{'=' * 60}")
    print(f"EXPERIMENT: {name}")
    print(f"{'=' * 60}")

    model = DustModel(encoder_type='lstm', fusion='attention',
                      embed_dim=256, dropout=0.24, num_heads=8).to(device)

    criterion_fl = FocalLoss(alpha=alpha, gamma=gamma)

    model, probs, preds, true, history, threshold = train_and_evaluate(
        model, train_dl, val_dl, test_dl,
        criterion_fl, lr=0.00032, weight_decay=0.003,
        epochs=20, model_name=name
    )

    nn_results[name] = {'probs': probs, 'preds': preds, 'model': model}
    all_histories[name] = history
    all_thresholds[name] = threshold

    safe = name.lower().replace(' ', '_').replace('+', '_').replace('=', '').replace('.', '')
    torch.save(model.state_dict(), f'/kaggle/working/models/{safe}_v2.pt')
    print(f"  Saved: {safe}_v2.pt")

    plot_model_performance(
        history, true, probs, preds, name,
        save_prefix=f'/kaggle/working/{safe}_v2'
    )

    torch.cuda.empty_cache()
    gc.collect()

## 9. Complete Results

In [ ]:
print("COMPLETE RESULTS COMPARISON (v2: AOD threshold 0.7)")

print(f"\n{'Model':<35} {'P':>6} {'R':>6} {'F1':>6} {'Acc':>6} {'AUC':>6} {'PR-AUC':>7} {'MCC':>6} {'Thr':>5}")

all_results = {}
all_scores = {}

for name, probs, preds in [
    ('Logistic Regression', lr_probs, lr_preds),
    ('Random Forest', rf_probs, rf_preds),
    ('XGBoost', xgb_probs, xgb_preds),
]:
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    prauc = average_precision_score(y_test, probs)
    mcc = matthews_corrcoef(y_test, preds)
    print(f"{name:<35} {p:>6.4f} {r:>6.4f} {f:>6.4f} {acc:>6.4f} {auc:>6.4f} {prauc:>7.4f} {mcc:>6.4f} {'0.50':>5}")
    all_results[name] = {'probs': probs, 'preds': preds}
    all_scores[name] = {'P': p, 'R': r, 'F1': f, 'Acc': acc, 'AUC': auc, 'PR-AUC': prauc, 'MCC': mcc}

for name, data in nn_results.items():
    probs, preds = data['probs'], data['preds']
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    prauc = average_precision_score(y_test, probs)
    mcc = matthews_corrcoef(y_test, preds)
    thr = all_thresholds.get(name, 0.5)
    print(f"{name:<35} {p:>6.4f} {r:>6.4f} {f:>6.4f} {acc:>6.4f} {auc:>6.4f} {prauc:>7.4f} {mcc:>6.4f} {thr:>5.2f}")
    all_results[name] = {'probs': probs, 'preds': preds}
    all_scores[name] = {'P': p, 'R': r, 'F1': f, 'Acc': acc, 'AUC': auc, 'PR-AUC': prauc, 'MCC': mcc}

## 10. Best Model Selection

In [ ]:
print("BEST MODEL SELECTION")

print("\nRanking by combined score (P=0.3, R=0.3, F1=0.2, PR-AUC=0.2):\n")

rankings = []
for name, scores in all_scores.items():
    combined = (0.3 * scores['P'] + 0.3 * scores['R'] +
                0.2 * scores['F1'] + 0.2 * scores['PR-AUC'])
    rankings.append((combined, name, scores))

rankings.sort(reverse=True)

print(f"{'Rank':>4} {'Model':<35} {'Score':>7} {'P':>6} {'R':>6} {'F1':>6} {'PR-AUC':>7}")
print("-" * 80)
for rank, (score, name, s) in enumerate(rankings, 1):
    marker = " <-- BEST" if rank == 1 else ""
    print(f"{rank:>4} {name:<35} {score:>7.4f} {s['P']:>6.4f} {s['R']:>6.4f} "
          f"{s['F1']:>6.4f} {s['PR-AUC']:>7.4f}{marker}")

best_name = rankings[0][1]
best_scores = rankings[0][2]

print(f"\nBest model: {best_name}")
for metric, val in best_scores.items():
    print(f"  {metric}: {val:.4f}")

if best_name in nn_results and 'model' in nn_results[best_name]:
    torch.save(nn_results[best_name]['model'].state_dict(),
               '/kaggle/working/models/best_model_v2.pt')
    print(f"\nSaved as: best_model_v2.pt")
elif best_name == 'XGBoost':
    joblib.dump(best_xgb, '/kaggle/working/models/best_model_v2_xgb.pkl')
elif best_name == 'Random Forest':
    joblib.dump(best_rf, '/kaggle/working/models/best_model_v2_rf.pkl')

np.savez('/kaggle/working/models/best_model_v2_predictions.npz',
         probs=all_results[best_name]['probs'],
         preds=all_results[best_name]['preds'],
         y_test=y_test)
print("Saved: best_model_v2_predictions.npz")

## 11. Threshold Calibration (All Models at P >= 0.6)

In [ ]:
print("THRESHOLD CALIBRATION: PRECISION >= 0.6 TARGET")

print(f"\n{'Model':<35} {'Threshold':>10} {'P':>7} {'R':>7} {'F1':>7} {'Acc':>7}")
print("-" * 75)

for name, data in all_results.items():
    probs = data['probs']
    best_f1 = 0
    best_t, best_p, best_r, best_acc = 0.5, 0, 0, 0

    for t in np.arange(0.1, 0.95, 0.01):
        preds = (probs > t).astype(int)
        if preds.sum() == 0:
            continue
        p = precision_score(y_test, preds, zero_division=0)
        r = recall_score(y_test, preds, zero_division=0)
        f = f1_score(y_test, preds, zero_division=0)
        acc = accuracy_score(y_test, preds)
        if p >= 0.6 and f > best_f1:
            best_f1 = f
            best_t = t
            best_p = p
            best_r = r
            best_acc = acc

    if best_p >= 0.6:
        print(f"{name:<35} {best_t:>10.2f} {best_p:>7.4f} {best_r:>7.4f} {best_f1:>7.4f} {best_acc:>7.4f}")
    else:
        print(f"{name:<35} {'Cannot reach 0.6':>40}")

## 12. Overfitting Analysis and Feature Importance

In [ ]:
n_plots = len(all_histories)
n_cols = 3
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
fig.suptitle('Training vs Validation Loss (v2: AOD 0.7)', fontsize=14, fontweight='bold')

for ax, (name, history) in zip(axes.flatten(), all_histories.items()):
    ax.plot(history['train_loss'], label='Train', color='#1A5FA0', lw=2)
    ax.plot(history['val_loss'], label='Val', color='#B84020', lw=2)
    ax.set_title(name, fontsize=9, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
for ax in axes.flatten()[n_plots:]:
    ax.set_visible(False)
plt.tight_layout()
plt.savefig('overfitting_v2.png', dpi=150, bbox_inches='tight')
plt.show()

# Feature importance
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.suptitle('Feature Importance (v2)', fontsize=13, fontweight='bold')
for ax, (name, model, color) in zip(axes, [
    ('Random Forest', best_rf, '#A06C10'),
    ('XGBoost', best_xgb, '#4A42B8')]):
    imp = model.feature_importances_
    idx_sort = np.argsort(imp)
    ax.barh([feature_names[i] for i in idx_sort], imp[idx_sort], color=color, alpha=0.85)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Importance Score')
    ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('feature_importance_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Save All Models

In [ ]:
print("All saved models:")
for f in sorted(os.listdir('/kaggle/working/models')):
    size = os.path.getsize(f'/kaggle/working/models/{f}') / 1e6
    print(f"  {f}: {size:.1f} MB")
print(f"\nBest model: {best_name}")